# 🧬 Embryo Fragmentation — YOLOv8 Training
### step by step karna h 
##### Comment Hindi me taki next time mujhe samjne me asani ho
---

## ✅ STEP 1 — GPU check karo
Pehle confirm karo ki Colab GPU de raha hai. **Runtime → Change runtime type → T4 GPU** select karo pehle!

In [ ]:
# GPU check
!nvidia-smi
print('\n✅ GPU mil gayi! Aage badho.')

## ✅ STEP 2 — Ultralytics install karo

In [ ]:
!pip install ultralytics -q
import ultralytics
print(f'✅ Ultralytics version: {ultralytics.__version__}')

## ✅ STEP 3 — Google Drive mount karo
Yahan ek popup aayega — **Allow** karo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mount ho gayi!')

## ✅ STEP 4 — Roboflow se dataset download karo

**Pehle Roboflow pe yeh karo:**
1. roboflow.com - apna project kholo
2. **Versions** - Version 5
3. **Export Dataset** click karo
4. Format: **YOLOv8** (Segmentation wala)
5. **Show Download Code** click karo → ek `rf.download(...)` line milegi
6. Woh API key aur project name neeche paste karo

## in sab step ke alawa usko local me download karke mtrain kiya tha maine wo asan tha 

⚠️ **`YOUR_API_KEY` aur project details apne Roboflow se copy karna!**

In [ ]:
!pip install roboflow -q

from roboflow import Roboflow

# ⬇️ Roboflow se copy karo apna code — yahan paste karo
rf = Roboflow(api_key="IZnutMkVcazdHBQWvIvw")   # Roboflow API key
project = rf.workspace("mohammad-dannish").project("section-of-best-embryo")
version = project.version(5)
dataset = version.download("yolov8-seg")   # segmentation format

print(f'\n✅ Dataset download ho gaya!')
print(f'📁 Path: {dataset.location}')

## ✅ STEP 5 — Dataset verify karo
Check karo ki train/valid/labels sab aaye hain

In [ ]:
import os

base = dataset.location

for split in ['train', 'valid', 'test']:
    img_path = os.path.join(base, split, 'images')
    lbl_path = os.path.join(base, split, 'labels')
    imgs = len(os.listdir(img_path)) if os.path.exists(img_path) else 0
    lbls = len(os.listdir(lbl_path)) if os.path.exists(lbl_path) else 0
    status = '✅' if imgs > 0 and lbls > 0 else '❌ PROBLEM!'
    print(f'{status} {split:6s} → Images: {imgs:4d} | Labels: {lbls:4d}')

# data.yaml print karo
yaml_path = os.path.join(base, 'data.yaml')
print('\n--- data.yaml ---')
with open(yaml_path) as f:
    print(f.read())

# Classes check
import yaml
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

print(f"\nClasses: {cfg['names']}")
if cfg['nc'] == 2 and 'Embryo' in str(cfg['names']) and 'Fragmentation' in str(cfg['names']):
    print('✅ Classes sahi hain — Embryo(0) aur Fragmentation(1)')
else:
    print('❌ Classes galat hain! Roboflow pe check karo.')

## ✅ STEP 6 — Training shuru karo! 🚀
Yeh 20-30 min lagega. Chalane ke baad chhod do — Colab train karta rahega.

In [ ]:
from ultralytics import YOLO
import os

yaml_path = os.path.join(dataset.location, 'data.yaml')

# Pretrained segmentation model se shuru karo
model = YOLO('yolov8n-seg.pt')

results = model.train(
    data    = yaml_path,
    epochs  = 100,
    imgsz   = 640,
    task    = 'segment',    # ZAROOR — segmentation mask chahiye
    batch   = 8,
    patience= 20,           # 20 epochs mein improve na ho to early stop
    name    = 'embryo_v2',
    augment = True,         # data augmentation ON
    plots   = True,         # training graphs save karega
)

print('\n🎉 Training complete!')
print(f'Best model: {results.save_dir}/weights/best.pt')

## ✅ STEP 7 — Model test karo (ek image pe)
Dekho ki model sahi detect kar raha hai

In [ ]:
import glob
from ultralytics import YOLO
from PIL import Image
import matplotlib.pyplot as plt

# Naya trained model load karo
best_pt = glob.glob('/content/runs/segment/embryo_v2*/weights/best.pt')[0]
print(f'Loading: {best_pt}')
model = YOLO(best_pt)

# Test image uthao
test_imgs = glob.glob(os.path.join(dataset.location, 'test/images/*.jpg'))
if not test_imgs:
    test_imgs = glob.glob(os.path.join(dataset.location, 'valid/images/*.jpg'))

test_img = test_imgs[0]
print(f'Testing on: {os.path.basename(test_img)}')

# Predict karo
results = model.predict(test_img, conf=0.25, retina_masks=True, verbose=False)
res = results[0]

# Classes count karo
if res.boxes is not None:
    classes = res.boxes.cls.cpu().numpy().astype(int)
    embryo_count = (classes == 0).sum()
    frag_count   = (classes == 1).sum()
    print(f'\n✅ Detected:')
    print(f'   Embryo:        {embryo_count}')
    print(f'   Fragmentation: {frag_count}')
    if frag_count == 0:
        print('⚠️  Fragment detect nahi hua — confidence lower karo ya zyada train data chahiye')
else:
    print('❌ Kuch detect nahi hua')

# Plot karo
annotated = res.plot()
plt.figure(figsize=(8,8))
plt.imshow(annotated[:,:,::-1])
plt.axis('off')
plt.title('Detection Result')
plt.show()

## ✅ STEP 8 — best.pt Google Drive mein save karo
Taaki baad mein use kar sako

In [ ]:
import shutil, glob

best_pt = glob.glob('/content/runs/segment/embryo_v2*/weights/best.pt')[0]

# Drive mein save karo
save_path = '/content/drive/MyDrive/embryo_best_v2.pt'
shutil.copy(best_pt, save_path)

print(f'✅ Model save ho gaya!')
print(f'📁 Location: {save_path}')
print(f'\nAb yeh file apne PC pe download karo aur app ke folder mein best.pt ke naam se replace karo!')

## ✅ STEP 9 — app.py update karo
Naye model ke liye class names match karna hai

In [ ]:
# Yeh sirf reminder hai — app.py mein yeh constants confirm karo:
reminder = """
app.py mein yeh values honi chahiye:

CLASS_EMBRYO = 0        # 'Embryo'
CLASS_FRAG   = 1        # 'Fragmentation'
CONF_EMBRYO  = 0.30     # thoda lower kiya
CONF_FRAG    = 0.20     # fragment pakdne ke liye
"""
print(reminder)